# Fine-tuning ModernBERT for Emotion Classification

| Colab | GitHub |
|---|---|
| <a target="_blank" href="https://colab.research.google.com/github/unionai/workshops/blob/main/tutorials/bert-fine-tuning-emotion/bert-emotion-tutorial.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> | [Full code and repo](https://github.com/unionai/workshops/tree/main/tutorials/bert-fine-tuning-emotion) |

Fine-tune [ModernBERT](https://huggingface.co/answerdotai/ModernBERT-base) to classify emotions in text, then **explore how the model makes decisions** with attention heatmaps and gradient-based token attribution. Everything runs on GPUs with [Modal](https://modal.com).

### What we'll build

```
┌──────────┐    ┌────────────┐    ┌────────────┐    ┌─────────────────┐
│ Get Data │───▶│   Train    │───▶│  Evaluate  │───▶│    Explore      │
│  (CPU)   │    │   (GPU)    │    │   (GPU)    │    │   Inference     │
└──────────┘    └────────────┘    └────────────┘    │    (GPU)        │
 emotion         ModernBERT        Confusion        └─────────────────┘
 dataset         fine-tuning       matrix +           Attention heatmaps
                 with loss/         per-class          + token importance
                 eval charts        metrics            + misclassification
                                                       analysis
```

### What makes this interesting

Beyond just training a classifier, we'll look **inside the model** to understand:
- **Attention heatmaps**: which words does the model "look at" when classifying?
- **Token importance**: which words actually *drive* the prediction? (gradient attribution)
- **Misclassification analysis**: where does the model fail, and why?
- **Negation handling**: the dataset lacks negated examples, so "I am NOT angry" still predicts anger. We'll see why.

Then we'll **deploy the model** as a FastAPI endpoint with a **Gradio frontend** for interactive exploration.

### The dataset

We use [dair-ai/emotion](https://huggingface.co/datasets/dair-ai/emotion), which contains ~20k English Twitter messages labeled with 6 emotions:

| Label | Emotion | Example |
|-------|---------|--------|
| 0 | sadness | "i feel so empty inside" |
| 1 | joy | "i am so happy right now" |
| 2 | love | "i feel blessed to have you" |
| 3 | anger | "i am furious about this" |
| 4 | fear | "i feel so scared and anxious" |
| 5 | surprise | "i cant believe this just happened" |


---

## Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !git clone https://github.com/unionai/workshops
    %cd workshops/tutorials/bert-fine-tuning-emotion
    !pip install -r requirements.txt
    !pip install pygments

from utils.file_viewer import view_file


## Connect to Modal

Authenticate once. This opens a browser to link your Modal account (sign up free at [modal.com](https://modal.com)). Modal provisions the CPU/GPU containers for you — there's no cluster to manage.

In [ ]:
!modal setup


### Set your HuggingFace token (optional)

ModernBERT isn't gated, so you can skip this. If you swap to a gated model, store your token as a **Modal secret** named `huggingface-secret` — the pipeline functions read it as `HF_TOKEN` wherever they run in the cloud.

Even for ungated models, create the secret once (a placeholder value is fine) so the functions can start.

In [ ]:
import os
from getpass import getpass

# Optional — only needed for gated models
hf_token = os.environ.get('HF_TOKEN', '')
# hf_token = getpass('HF_TOKEN: ')
!modal secret create huggingface-secret HF_TOKEN=$hf_token --force


---

## Run the Pipeline

We'll walk through the code below while our model trains.

### Pipeline parameters

| Flag | Default | Description |
|------|---------|-------------|
| `--model-name` | `answerdotai/ModernBERT-base` | HuggingFace encoder model |
| `--epochs` | `3` | Training epochs |
| `--lr` | `2e-5` | Learning rate |
| `--batch-size` | `16` | Batch size |
| `--max-train-samples` | `10000` | Training examples |
| `--max-eval-samples` | `2000` | Eval examples |
| `--num-eval-examples` | `200` | Examples for base vs fine-tuned comparison |
| `--num-explore-examples` | `12` | Examples for attention/attribution analysis |

`modal run` builds the container image (first run only), then runs every step in the cloud — `get_data` on CPU, `train`/`evaluate`/`explore_inference` on a T4 GPU. The fine-tuned model and the HTML reports are written to `outputs/` locally when it finishes.

In [ ]:
!modal run workflow.py \
    --epochs 3 \
    --max-train-samples 4000 \
    --num-eval-examples 200 \
    --num-explore-examples 12


### Smaller/faster run

Shrink the dataset and epochs to verify everything end to end in a few minutes:

```bash
modal run workflow.py \
    --max-train-samples 200 \
    --max-eval-samples 50 \
    --epochs 1 \
    --num-eval-examples 30 \
    --num-explore-examples 6
```

---

## Code Walkthrough

Let's look at the key files. The pipeline has 4 functions chained together, each building a rich visual HTML report.

### Project config

`config.py` defines the shared `modal.Image` (all deps installed inside the container), the `modal.App`, the HuggingFace secret, and two `modal.Volume`s that pass the dataset and the fine-tuned model between steps.

In [ ]:
view_file("config.py")


### Local dependencies

The only thing installed locally is `modal` — torch, transformers, datasets, and the rest live in the `modal.Image`.

In [ ]:
view_file("requirements.txt")


### The workflow

The full pipeline is in `workflow.py`. Each step is an `@app.function` requesting exactly the resources it needs (CPU vs T4 GPU). Let's look at each task.

In [ ]:
view_file("workflow.py")


#### Key things to notice in the workflow:

**Training function (`train`):**
- Uses `AutoModelForSequenceClassification` with `num_labels=6`, on a T4 GPU (`gpu="T4"`)
- A `ReportCallback` hooks into the HuggingFace `Trainer` to collect loss and eval metrics for the charts
- Eval runs at each epoch boundary so the report shows accuracy improving
- Saves the fine-tuned model to the `bert-emotion-model` volume so evaluate/explore/serve can read it back

**Evaluation function (`evaluate`):**
- Compares base model (random classifier head) vs fine-tuned
- Generates confusion matrix heatmap, per-class precision/recall/F1
- Per-class accuracy bar chart (base vs fine-tuned)

**Explore inference function (`explore_inference`):**
- Loads model with `attn_implementation="eager"` to extract attention weights (flash attention doesn't return them)
- **Attention heatmap**: CLS token attention from the last transformer layer, averaged across heads
- **Token importance**: gradient x embedding attribution, hooks into the embedding layer to compute gradients
- **Misclassification spotlight**: sorts wrong predictions by confidence to find blind spots

**Reports without a live panel:** Modal has no live-report panel, so each function *returns* its HTML and the `local_entrypoint` writes it to `outputs/*_report.html` — open the files in a browser after the run.

### Report helpers

All visualizations are pure SVG/HTML with no matplotlib dependency. This includes:
- Line charts (training loss, eval metrics)
- Bar charts (per-class accuracy comparison)
- Confusion matrix heatmap
- Colored text for attention/importance visualization
- Confidence bars for emotion scores

In [ ]:
view_file("report_helpers.py")


---

## Understanding the Results

Open the reports written to `outputs/` (`pipeline_report.html`, `train_report.html`, `evaluate_report.html`, `explore_report.html`). Here's what to look for:

### Training report
- **Loss curve**: should decrease steadily. If it plateaus early, the model may need more data or a different learning rate.
- **Eval accuracy/F1**: updates after each epoch. Expect ~90%+ accuracy on emotion classification with ModernBERT.

### Evaluation report
- **Confusion matrix**: look at which emotions get confused. Common: anger↔fear, love↔joy.
- **Per-class metrics**: some emotions (joy, sadness) are easier to classify than others (love, surprise).
- **Base vs fine-tuned bar chart**: the base model has a random classifier head, so ~16.7% accuracy (1/6). Fine-tuned should be dramatically better.

### Explore inference report
- **Attention heatmaps**: darker tokens = more attention from [CLS]. The model should focus on emotional words ("happy", "terrified", "love").
- **Token importance**: green = supports prediction, red = opposes. Look for cases where function words unexpectedly matter.
- **Misclassification spotlight**: the most confident wrong predictions reveal the model's blind spots.

### Inspect runs in the dashboard

Every `modal run` streams logs to your terminal and records the run in the [Modal dashboard](https://modal.com/apps), where you can browse past executions, logs, and per-function container metrics (GPU utilization, memory, duration). List your apps with:

```bash
modal app list
```

or visit [modal.com/apps](https://modal.com/apps) directly.

---

## How Attention Visualization Works

BERT-style models use **multi-head self-attention** where each token attends to every other token, producing an attention weight matrix. Here's what we extract:

```
Input:  [CLS] i am so happy right now [SEP]

Last layer attention (averaged across heads):

[CLS] → i(0.05) am(0.08) so(0.15) happy(0.45) right(0.04) now(0.12) [SEP](0.11)
                                    ^^^^^
                            highest attention = most relevant for classification
```

The **[CLS] token** is special. Its final representation is fed to the classifier head. So the [CLS] attention pattern tells us: *"what did the model look at when making its classification decision?"*

### Why `attn_implementation="eager"`?

ModernBERT uses **Flash Attention** by default for speed, but Flash Attention doesn't return attention weight matrices. We switch to eager (standard) attention for the explore_inference step to extract the weights. This is slower but only runs on a few examples.

### Gradient-based token importance

Attention shows where the model *looks*, but not necessarily what *drives* the prediction. For that, we use gradient attribution:

```python
importance(token) = ||grad(prediction, embedding(token)) * embedding(token)||
```

This measures how much each token's embedding *actually influences* the predicted class score. It's complementary to attention. Sometimes a token gets high attention but low importance, and vice versa.

---

## Serve the Model

Once the pipeline completes, deploy the fine-tuned model as a live API. Both `serve.py` and `app_gradio.py` read the model from the `bert-emotion-model` volume, so run the pipeline first.

### Step 1: The FastAPI server

`serve.py` loads the model once per container (via `@modal.enter()` on an `@app.cls`) and exposes a FastAPI app with `@modal.asgi_app()`. The `/predict` endpoint returns:
- Predicted emotion + confidence
- Full probability distribution across all 6 emotions
- Attention weights per token (for heatmap visualization)

Use `modal serve` for a hot-reloading dev server (ephemeral URL) or `modal deploy` for a persistent URL.

In [ ]:
# Persistent deployment (prints a stable URL)
!modal deploy serve.py

# Or a dev server with hot-reload (ephemeral URL, tails logs):
# !modal serve serve.py


In [ ]:
view_file("serve.py")


### Test the endpoint

In [ ]:
%%bash
# Replace with the URL printed by modal deploy/serve
curl -X POST https://your-app-url/predict \
  -H "Content-Type: application/json" \
  -d '{"text": "I am so happy today!"}'


### Step 2: The Gradio frontend

`app_gradio.py` serves an interactive UI with `@modal.asgi_app()` + `mount_gradio_app`. It loads the fine-tuned model from the shared volume and runs inference in-process, so it works on its own once training has produced a model. `modal serve` prints a URL you can open in the browser.

In [ ]:
# Persistent deployment
!modal deploy app_gradio.py

# Or a dev server with hot-reload:
# !modal serve app_gradio.py


In [ ]:
view_file("app_gradio.py")


---

## Try It: Explore the Model's Behavior

Once the Gradio app is running, try these inputs and observe the attention patterns:

### Straightforward emotions
These should get high confidence with attention focused on the emotional keywords:
- `"I am so happy right now"` → joy, attention on "happy"
- `"This makes me furious"` → anger, attention on "furious"
- `"I love you more than anything"` → love, attention on "love"

### Negation (a dataset gap)
Try these and watch the model fail:
- `"this does not make me angry"` → predicts anger (~99%!)
- `"I am not sad anymore"` → likely predicts sadness
- `"I'm not surprised at all"` → likely predicts surprise

Look at the attention heatmap. The model attends to "angry", "sad", "surprised" but doesn't properly handle the negation. This isn't a limitation of the model architecture itself. The training data (Twitter messages) is mostly direct emotional statements and rarely contains negated examples, so the model simply pattern-matches on emotional keywords. A dataset with more negated examples would teach the model to handle these correctly.

### Ambiguous / mixed emotions
These are interesting because the confidence distribution spreads across multiple emotions:
- `"I can't believe they did that to me"` → anger or surprise?
- `"I'm nervous but excited about tomorrow"` → fear or joy?
- `"I miss you so much"` → sadness or love?

### Subtle / complex
- `"The meeting went fine"` → low confidence, ambiguous
- `"Whatever"` → what does the model do with minimal signal?

---

## Why ModernBERT?

[ModernBERT](https://huggingface.co/answerdotai/ModernBERT-base) (2024) is a drop-in replacement for BERT-base with modern improvements:

| Feature | BERT (2018) | ModernBERT (2024) |
|---------|-------------|-------------------|
| Context length | 512 tokens | 8,192 tokens |
| Position encoding | Absolute | Rotary (RoPE) |
| Attention | Standard | Flash Attention |
| Parameters | ~110M | ~150M |
| Training data | BooksCorpus + Wikipedia | 2T tokens, diverse sources |

Same `AutoModelForSequenceClassification` API, better results. The attention visualization works identically since it's still a multi-head transformer encoder.

You can swap to classic BERT anytime:
```bash
modal run workflow.py --model-name "bert-base-uncased"
```

---

## Key Takeaways

**Fine-tuning:**
- Encoder models (BERT/ModernBERT) are excellent for classification. Fast to train, small to deploy.
- `AutoModelForSequenceClassification` + HuggingFace `Trainer` handles most of the complexity
- A random classifier head gets ~16% accuracy. A few epochs of fine-tuning gets 90%+.

**Model interpretability:**
- Attention heatmaps show *where* the model looks (CLS token attention pattern)
- Gradient attribution shows *what drives* the prediction (which tokens influence the score)
- These are complementary. A token can have high attention but low importance.
- Misclassification analysis reveals systematic blind spots (negation, ambiguity)

**Modal:**
- Each `@app.function` requests exactly the resources it needs — CPU for data prep, T4 GPU for training and inference
- `modal.Volume` passes the dataset and model artifacts between steps, and on to the serving apps
- Dependencies live in the `modal.Image`, so the only local requirement is `modal`
- The same fine-tuned model serves in production via `modal deploy serve.py` (FastAPI) and `modal deploy app_gradio.py` (Gradio), both `@modal.asgi_app()` web endpoints

---

## Resources

- Full code: [tutorials/bert-fine-tuning-emotion](https://github.com/unionai/workshops/tree/main/tutorials/bert-fine-tuning-emotion)
- Get started with Modal: [modal.com/docs](https://modal.com/docs)
- Modal dashboard: [modal.com/apps](https://modal.com/apps)
- ModernBERT: [huggingface.co/answerdotai/ModernBERT-base](https://huggingface.co/answerdotai/ModernBERT-base)
- Emotion dataset: [huggingface.co/datasets/dair-ai/emotion](https://huggingface.co/datasets/dair-ai/emotion)